# 读出 PDM 区间与边缘密度

本例使用 0.1.1 候选源码，默认计算内置的合成 `P_ADO`。已有结果时，把 `result_dir` 改成 GUI 或 CLI 保存完成的结果目录，即包含 `manifest.json` 和 `density.npz` 的目录。`Path` 可使用绝对路径，也可使用相对当前 Notebook 的路径。

Run the bundled synthetic input or set `result_dir` to a completed GUI/CLI snapshot.

In [ ]:
%matplotlib inline
from pathlib import Path

from multipolepdm.io.export import load_result
from multipolepdm.io.input import example_data, job_from_dict
from multipolepdm.jobs.workers import compute_job
from multipolepdm.pdm.statistics import summarize_density
from multipolepdm.plotting import make_preview

result_dir = None  # Example: Path('pdm-results/my-result')
if result_dir is None:
    job = job_from_dict(example_data('P_ADO'))
    result, summary = compute_job(job)
else:
    result = load_result(result_dir)
    summary = summarize_density(result)

print(result.job.mode, result.job.label)
print('Source spin I =', result.response.reference_spin)
print('Finite-scan normalization =', result.normalization)
for warning in result.warnings:
    print(warning)

角空间先求蓝色 68% HDI、橙色 `q16/q50/q84` 与灰色均值和标准差。逐段查看 HDI 的端点、峰位范围和质量。各段质量合计 68%。`Summary.delta` 对角端点和峰位取正切，HDI 的选择依据仍是角密度。灰线按角密度积分和实际网格宽度得到，取正切后的中心是角均值的变换，灰线之间的概率质量由分布决定。若某个变换端点越界，查看 `mean_bound_flags`。宽度的统计直接在 `sigma/I` 密度上计算，`I` 取设定高斯布居的初态自旋。

图题 `Full Range Marginalization; Constant Weight` 表示扫描全域边缘化与常数附加权重。观测图和二维参数图分别使用 `viridis` 色标。角轴显示 π 分数。

In [ ]:
for name, marginal in (
    ('arctan(delta) [rad]', summary.arctan_delta),
    ('delta (transformed angle estimates)', summary.delta),
    ('sigma/I', summary.sigma_over_i),
):
    interval = marginal.intervals[0]
    print('\n' + name)
    print('ETI 68%  q16, q50, q84:', interval.eti[0], marginal.median, interval.eti[1])
    for number, segment in enumerate(interval.hdi_segments, start=1):
        print(f'HDI 68%  segment {number}:', segment.bounds,
              'mass =', segment.mass, 'mode ranges =', segment.mode_ranges)
    print('Total HDI mass:', sum(segment.mass for segment in interval.hdi_segments))
    print('Mean-based center and bounds:', marginal.mean, marginal.mean_bounds)
    if marginal.std is not None:
        print('SD:', marginal.std)
    if any(marginal.mean_bound_flags):
        print('Mean bound flags:', marginal.mean_bound_flags)

make_preview(result, summary).figure